# Ayurveda / Phytochemical — KG Processing

**Outputs produced:**

| File | Relation | Source |
|------|----------|--------|
| `Phytodata_Chemical_Disease.csv` | ChemicalEntity_Disease | Phyto Data-1 |
| `Phytodata_Plantspecies_Chemical.csv` | PlantSpecies_ChemicalEntity | Phyto Data-1 (species–chemical) |
| `Phytodata_Plantspecies_Disease.csv` | PlantSpecies_Disease | Phyto Data-1 (species–disease) |
| `Phytodata_PlantSpecies_Chemical_2.csv` | PlantSpecies_ChemicalEntity | Phyto Data-2 (GitHub) |
| `Phytodata_PlantSpecies_Chemical_3.csv` | PlantSpecies_ChemicalEntity | Phyto Data-3 (all_species_compound_species) |
| `Phytodata_Ayurvedic_PlantSpecies_Disease.csv` | PlantSpecies_Disease | Ayurvedic Data-1 |
| `Phytodata_Ayurvedic_PlantSpecies_Chemical.csv` | PlantSpecies_ChemicalEntity | Ayurvedic Data-2 |
| `Phytodata_IMPPAT_PlantSpecies_Chemical.csv` | PlantSpecies_ChemicalEntity | IMPPAT Data-1 |
| `Phytodata_IMPPAT_PlantSpecies_Disease.csv` | PlantSpecies_Disease | IMPPAT Data-2 |
| `Phytodata_PhytoHUB_PlantSpecies_Chemical.csv` | PlantSpecies_ChemicalEntity | PhytoHUB |
| `Phytodata_TTD_Chemical_Disease.csv` | ChemicalEntity_Disease | TTD |

---




## 0. Path Configuration

In [2]:
import os

# ── Edit only these four lines ───────────────────────────────────────────────
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
DB_BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/"
AYUSHI_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/Ayurveda/"   # Ayurveda / phytochemical raw files
OUT_PATH     = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"      # all output KG CSVs written here
# ─────────────────────────────────────────────────────────────────────────────

# Sub-paths (derived automatically)
PHYTO_SORTED_PATH = os.path.join(AYUSHI_PATH, "Sorted_Triples")
PHYTOHUB_PATH     = os.path.join(BASE_PATH, "phytohub")
IMPPAT_PATH       = os.path.join(BASE_PATH, "immpat")
TTD_PATH          = os.path.join(AYUSHI_PATH, "TTD")

os.makedirs(OUT_PATH, exist_ok=True)
print( AYUSHI_PATH)
print( OUT_PATH)


/storage/Arushi/090526_EvoAge/kg_formation/data_collection/Ayurveda/
/storage/Arushi/090526_EvoAge/kg_formation/processed_data/


## 1. Imports

In [3]:
import os
import re
import numpy as np
import pandas as pd


## 2. Shared Helpers

In [4]:
KG_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'tail_type',
    'head_detail_name', 'tail_detail_name',
    'head_id_is', 'tail_id_is',
    'relation_source',
]

def reorder(df):
    """Return df keeping only KG_COLS that exist, in order."""
    return df[[c for c in KG_COLS if c in df.columns]]

def tail_id_is(series):
    """
    """
    return (
        series.fillna("")
        .apply(lambda x: "DOID_ID" if x.startswith("DOID")
                         else ("MESH_ID" if x else np.nan))
    )

def pubchem_id_is(series):
    """Return 'Pubchem' where value is numeric string, else NaN."""
    return np.where(series.astype(str).str.isnumeric(), "Pubchem", np.nan)

def strip_float(series):
    """Remove trailing .0 from numeric-looking strings."""
    return series.astype(str).str.replace(r'\.0$', '', regex=True)

def explode_tail(df, sep=',', strip_prefix='^human '):
    """Split Tail on sep, explode, strip whitespace, remove common prefixes."""
    df['Tail'] = df['Tail'].astype(str).str.split(sep)
    df = df.explode('Tail')
    df['Tail'] = df['Tail'].str.strip()
    if strip_prefix:
        df['Tail'] = df['Tail'].str.replace(strip_prefix, '', regex=True)
    return df


## 3. Reference Dictionaries

### 3.1 PubChem Synonym → CID (case-sensitive + lowercase)

In [5]:
Pubchem_Syn_fil = pd.read_csv(
    os.path.join(DB_BASE_PATH, "pubchem", "CID-Synonym-filtered"),
    sep="\t", header=None
)
Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
Pubchem_Syn_fil_dict_lower = {k.lower(): v for k, v in Pubchem_Syn_fil_dict.items()}
del Pubchem_Syn_fil
print(f"Pubchem_Syn_fil_dict       : {len(Pubchem_Syn_fil_dict):,}")
print(f"Pubchem_Syn_fil_dict_lower : {len(Pubchem_Syn_fil_dict_lower):,}")


Pubchem_Syn_fil_dict       : 102,877,691
Pubchem_Syn_fil_dict_lower : 102,877,659


### 3.2 PubChem CID → IUPAC Name / SMILES

In [6]:
Pubchem = pd.read_pickle(os.path.join(DB_BASE_PATH, "pubchem", "combined_df.pkl"))

Pubchem_CID_Smile_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_SMILES"]))
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_IUPAC_NAME"]))
Pubchem_IUPAC_CID_Dict_2= dict(zip(Pubchem["PUBCHEM_IUPAC_NAME"],   Pubchem["PUBCHEM_COMPOUND_CID"]))

del Pubchem
print(f"Pubchem_IUPAC_CID_Dict   : {len(Pubchem_IUPAC_CID_Dict):,}")
print(f"Pubchem_IUPAC_CID_Dict_2 : {len(Pubchem_IUPAC_CID_Dict_2):,}")


Pubchem_IUPAC_CID_Dict   : 119,177,440
Pubchem_IUPAC_CID_Dict_2 : 117,876,044


### 3.3 DrugBank ID → Name

In [7]:
Drugbank = pd.read_csv(
    os.path.join(DB_BASE_PATH, "drugbank", "ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv")
)
Drugbank_dict = dict(zip(Drugbank["drugbank_id"], Drugbank["name"]))
del Drugbank
print(f"Drugbank_dict : {len(Drugbank_dict):,}")


Drugbank_dict : 16,575


### 3.4 Disease Ontology (DOID)

In [8]:
DO_All_File = pd.read_csv(os.path.join(DB_BASE_PATH, "DO", "All_DO_ref_files.csv"))
DOID_disname_dict      = dict(zip(DO_All_File["id"],    DO_All_File["label"]))
DOID_disAltID_dict     = dict(zip(DO_All_File["id"],    DO_All_File["xrefs"]))
DOID_disname_DOID_dict = dict(zip(DO_All_File["label"], DO_All_File["id"]))
del DO_All_File
print(f"DOID_disname_DOID_dict : {len(DOID_disname_DOID_dict):,}")


DOID_disname_DOID_dict : 11,787


### 3.5 MedGen: MONDO / MeSH ID → CUI + preferred name

In [9]:
MedGen_CUID_Source_ID = pd.read_csv(
    os.path.join(DB_BASE_PATH, "MESH", "MeSH", "MedGenIDMappings.txt"), sep="\t"
)

# MONDO → CUI
MONDO_2_MESH = MedGen_CUID_Source_ID[
    MedGen_CUID_Source_ID["source_id"].str.contains("MONDO", na=False)
].copy()
MONDO_2_MESH_dict = dict(zip(MONDO_2_MESH["source_id"], MONDO_2_MESH["#CUI_or_CN_id"]))

# Lowercase disease name → CUI (for loose matching)
MONDO_2_MESH["pref_name_lower"] = MONDO_2_MESH["pref_name"].str.lower()
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH["pref_name_lower"], MONDO_2_MESH["#CUI_or_CN_id"]))

# MeSH source_id → preferred name
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(
    subset=["source_id", "source"], keep="first"
)
MedGen_MeshID_name_dict = dict(
    zip(MedGen_CUID_Source_ID["source_id"], MedGen_CUID_Source_ID["pref_name"])
)
del MedGen_CUID_Source_ID, MONDO_2_MESH
print(f"MONDOMESH_2_des_dict    : {len(MONDOMESH_2_des_dict):,}")
print(f"MedGen_MeshID_name_dict : {len(MedGen_MeshID_name_dict):,}")


MONDOMESH_2_des_dict    : 22,703
MedGen_MeshID_name_dict : 394,895


### 3.6 MeSH → DOID

In [10]:
Mesh2DOID = pd.read_csv(
    os.path.join(DB_BASE_PATH, "DO", "HumanDiseaseOntology-main", "DOreports", "MESHinDO.tsv"),
    sep="\t"
)
Mesh2DOID.rename(columns={"ID": "id", "LABEL": "label"}, inplace=True)
Mesh2DOID["xrefs"] = Mesh2DOID["xrefs"].str.split(", ")
Mesh2DOID = Mesh2DOID.explode("xrefs")
Mesh2DOID_dict = dict(zip(Mesh2DOID["xrefs"], Mesh2DOID["id"]))
del Mesh2DOID
print(f"Mesh2DOID_dict : {len(Mesh2DOID_dict):,}")


Mesh2DOID_dict : 3,687


### 3.7 MeSH Combined (ID → Name)

In [11]:
MESH = pd.read_csv(os.path.join(DB_BASE_PATH, "MESH", "MESH_Combined.csv"))
MESH_dict      = dict(zip(MESH["ID"],   MESH["Name"]))
MESH_name_dict = dict(zip(MESH["Name"], MESH["ID"]))
del MESH
print(f"MESH_dict : {len(MESH_dict):,}")


MESH_dict : 38,520


### 3.8 MeSH Supplementary (ID → Name)

In [12]:
Mesh_supp = pd.read_csv(
    os.path.join(DB_BASE_PATH, "MESH", "Mesh_Supplementary_Info.csv")
)
Mesh_supp_dict = dict(zip(Mesh_supp["ID"], Mesh_supp["Name"]))
del Mesh_supp
print(f"Mesh_supp_dict : {len(Mesh_supp_dict):,}")


Mesh_supp_dict : 324,150


### 3.9 Shared disease-Tail resolver
Maps a raw Tail disease string (DOID label, MONDO name, or MeSH name) →
canonical disease ID, then swaps Tail ← ID and Tail_detail_name ← original label.


In [13]:
def resolve_disease_tail(df, tail_col="Tail"):
    """
    1. Explode comma-separated Tail strings.
    2. Map to DOID (via label) or MONDO CUI (via lowercase name).
    3. Swap: Tail ← disease ID; Tail_detail_name ← original string.
    4. Drop rows where disease could not be resolved.
    """
    df = explode_tail(df.copy())

    df["Tail_detail_name"] = (
        df[tail_col].map(DOID_disname_DOID_dict)
        .fillna(df[tail_col].str.lower().map(MONDOMESH_2_des_dict))
    )
    df[[tail_col, "Tail_detail_name"]] = df[["Tail_detail_name", tail_col]]
    df = df[~df[tail_col].isna()].copy()
    return df


## §1 Phyto Data-1 — Chemical–Disease


In [94]:
# df1 = pd.read_csv(f'{BASE_PATH}other_sources/D1phyto_chemicals.db')
# df
import sqlite3
import pandas as pd

DB_PATH = f'{BASE_PATH}other_sources/D1phyto_chemicals.db'   # change path if needed
# OUT_PATH = "phyto_kg.csv"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM phyto_chemicals", conn)
conn.close()

# Build KG columns
# Head: IUPAC name; fall back to extract type, then phytochemical name
df["Head"] = (
    df["IUPAC_name"]
    .fillna(df["Type_of_phytochemicals/Extracts"])
    .fillna(df["Phytochemical_name/extract"])
)

df["Relation"]  = "Chemical_Disease"
df["Tail"]      = df["Type_of_cancer"]
df["Head_Type"] = "Chemical"
df["Tail_Type"] = "Disease"
df["Source"]    = df["ADDITIONAL_SOURCES_OF_INFORMATION"]

# Select final columns
COLS = [
    "Head", "Relation", "Tail",
    "Head_Type", "Tail_Type",
    "Source", "index",
    "Phytochemical_name/extract", "PMID"
]

out = df[COLS].reset_index(drop=True)

# Drop rows where both Head and Tail are missing
out = out.dropna(subset=["Head", "Tail"], how="all")

# out.to_csv(OUT_PATH, index=False)
# print(f"Saved {len(out)} rows → {OUT_PATH}")
# print(out.head(3).to_string())
out

,Head,Relation,Tail,Head_Type,Tail_Type,Source,index,Phytochemical_name/extract,PMID
0,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",Chemical_Disease,"human hepatoma, human breast cancer",Chemical,Disease,https://powo.science.kew.org/taxon/urn:lsid:ip...,0,Alpha-Turmerone,20438793.0
1,glycoalkaloid,Chemical_Disease,breast cancer,Chemical,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Sol...,1,Extract of Solanum torvum,34643844.0
2,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",Chemical_Disease,"human breast cancer, human ovarian cancer",Chemical,Disease,None,2,Mollugin,23065756.0
3,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,Chemical_Disease,breast cancer,Chemical,Disease,None,3,Triphenylethylene-Coumarin Hybrid 5c,30947677.0
4,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",Chemical_Disease,breast cancer,Chemical,Disease,None,4,(-)-Oleocanthal (OC),30781364.0
...,...,...,...,...,...,...,...,...,...
875,extract,Chemical_Disease,"lung cancer, human colorectal adenocarcinoma, ...",Chemical,Disease,https://pfaf.org/user/plant.aspx?latinname=Dio...,875,Diospyros lotus extract,19731038.0
876,essential oil,Chemical_Disease,"human breast adenocarcinoma, erythromyeloblast...",Chemical,Disease,https://powo.science.kew.org/taxon/urn:lsid:ip...,876,Dracocephalum surmandinum essential oil,20334155.0
877,5-(dithiolan-3-yl)pentanoic acid,Chemical_Disease,breast cancer,Chemical,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Ara...,877,Alpha-lipoic acid,20580704.0
878,essential oil,Chemical_Disease,breast cancer,Chemical,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Fer...,878,Essential Oil of Ferula assa-foetida,28900317.0


In [95]:
df1 = out

phyto_col = "Phytochemical_name/extract"
if phyto_col in df1.columns and "Head" in df1.columns:
    df1[["Head", phyto_col]] = df1[[phyto_col, "Head"]]

df1["Head"] = (
    df1["Head"].map(Pubchem_Syn_fil_dict)
    .fillna(df1.get(phyto_col, pd.Series(dtype=str)).map(Pubchem_Syn_fil_dict))
    .fillna(df1["Head"])
)
df1["Head"] = strip_float(df1["Head"])
df1

df1["Head_detail_name"] = df1["Head"].map(Pubchem_IUPAC_CID_Dict)

df1 = resolve_disease_tail(df1)
df1["Tail_type"] = "Disease"

df['Tail_id_is'] = np.where(
    df['Tail'].isna(),
    None,
    np.where(
        df['Tail'].str.startswith('DOID'),
        'DOID_ID',
        'MESH_ID'
    )
)

df['Head_id_is'] = np.where(
    df['Head'].astype(str).str.isnumeric(),
    'Pubchem',
    None
)
df1["head_type"]       = "ChemicalEntity"
df1["relation"]        = "ChemicalEntity_Disease"
df1["relation_source"] = df1.get("Source", "Phytochemical")

df1.columns = df1.columns.str.lower()
df1 = reorder(df1)
print(f"Chemical–Disease rows: {len(df1):,}")
df1

Chemical–Disease rows: 1,448


,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,relation_source
0,Alpha-Turmerone,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://powo.science.kew.org/taxon/urn:lsid:ip...
1,Extract of Solanum torvum,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://pfaf.org/user/Plant.aspx?LatinName=Sol...
2,124219,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",breast cancer,None
2,124219,Chemical_Disease,ChemicalEntity_Disease,DOID:2394,Chemical,ChemicalEntity,Disease,Disease,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",ovarian cancer,None
3,Triphenylethylene-Coumarin Hybrid 5c,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,None
...,...,...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus extract,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://pfaf.org/user/plant.aspx?latinname=Dio...
876,Dracocephalum surmandinum essential oil,Chemical_Disease,ChemicalEntity_Disease,DOID:3458,Chemical,ChemicalEntity,Disease,Disease,NaN,breast adenocarcinoma,https://powo.science.kew.org/taxon/urn:lsid:ip...
877,864,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,5-(dithiolan-3-yl)pentanoic acid,breast cancer,https://pfaf.org/user/Plant.aspx?LatinName=Ara...
878,Essential Oil of Ferula assa-foetida,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://pfaf.org/user/Plant.aspx?LatinName=Fer...


In [100]:
df1['relation_source'] = 'https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db'

In [101]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

In [102]:
df1.to_csv(os.path.join(OUT_PATH, "ttd/Phytodata_Chemical_Disease.csv"), index=False)
print("Saved: Phytodata_Chemical_Disease.csv")
# del df1

Saved: Phytodata_Chemical_Disease.csv


## §2 Phyto Data-1 — PlantSpecies–Chemical


In [109]:
import sqlite3
import pandas as pd

DB_PATH = f'{BASE_PATH}other_sources/D1phyto_chemicals.db'   # change path if needed
# OUT_PATH = "phyto_kg_plantspecies.csv"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM phyto_chemicals", conn)
conn.close()

# Build KG columns
df["Head"]      = df["Source_of_phytochemicals_(Name_of_Plant)"]
df["Relation"]  = "PlantSpecies_Chemical"
df["Tail"]      = (
    df["IUPAC_name"]
    .fillna(df["Type_of_phytochemicals/Extracts"])
    .fillna(df["Phytochemical_name/extract"])
)
df["Head_Type"] = "PlantSpecies"
df["Tail_Type"] = "Chemical"
df["Source"]    = df["ADDITIONAL_SOURCES_OF_INFORMATION"]

COLS = [
    "Head", "Relation", "Tail",
    "Head_Type", "Tail_Type",
    "Source", "index",
    "Phytochemical_name/extract", "PMID"
]

out = df[COLS].dropna(subset=["Head", "Tail"], how="all").reset_index(drop=True)
# out.to_csv(OUT_PATH, index=False)
# print(f"Saved {len(out)} rows → {OUT_PATH}")
# print(out.head(3).to_string())
out

,Head,Relation,Tail,Head_Type,Tail_Type,Source,index,Phytochemical_name/extract,PMID
0,Curcuma longa (turmeric),PlantSpecies_Chemical,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,Chemical,https://powo.science.kew.org/taxon/urn:lsid:ip...,0,Alpha-Turmerone,20438793.0
1,Solanum torvum (Pea Eggplant),PlantSpecies_Chemical,glycoalkaloid,PlantSpecies,Chemical,https://pfaf.org/user/Plant.aspx?LatinName=Sol...,1,Extract of Solanum torvum,34643844.0
2,Rubia cordifolia(Indian Madder),PlantSpecies_Chemical,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",PlantSpecies,Chemical,None,2,Mollugin,23065756.0
3,None,PlantSpecies_Chemical,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,Chemical,None,3,Triphenylethylene-Coumarin Hybrid 5c,30947677.0
4,Olea europaea (Olive),PlantSpecies_Chemical,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",PlantSpecies,Chemical,None,4,(-)-Oleocanthal (OC),30781364.0
...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_Chemical,extract,PlantSpecies,Chemical,https://pfaf.org/user/plant.aspx?latinname=Dio...,875,Diospyros lotus extract,19731038.0
876,Dracocephalum surmandinum,PlantSpecies_Chemical,essential oil,PlantSpecies,Chemical,https://powo.science.kew.org/taxon/urn:lsid:ip...,876,Dracocephalum surmandinum essential oil,20334155.0
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_Chemical,5-(dithiolan-3-yl)pentanoic acid,PlantSpecies,Chemical,https://pfaf.org/user/Plant.aspx?LatinName=Ara...,877,Alpha-lipoic acid,20580704.0
878,Ferula assa-foetida (Asafoetida),PlantSpecies_Chemical,essential oil,PlantSpecies,Chemical,https://pfaf.org/user/Plant.aspx?LatinName=Fer...,878,Essential Oil of Ferula assa-foetida,28900317.0


In [111]:
df2 = out
# phyto_col = "Phytochemical_name/extract"
# if phyto_col in df2.columns and "Tail" in df2.columns:
#     df2[["Tail", phyto_col]] = df2[[phyto_col, "Tail"]]

# df2["Tail"] = (
#     df2["Tail"].map(Pubchem_Syn_fil_dict)
#     .fillna(df2.get(phyto_col, pd.Series(dtype=str)).map(Pubchem_Syn_fil_dict))
#     .fillna(df2["Tail"])
# )

df2[['Phytochemical_name/extract', 'Tail']] = df2[['Tail', 'Phytochemical_name/extract']]
df2
df2['Tail'] = df2['Tail'].map(Pubchem_Syn_fil_dict).fillna(df2['Phytochemical_name/extract'].map(Pubchem_Syn_fil_dict)).fillna(df2['Tail'])
df2

df2["Tail"] = strip_float(df2["Tail"])
df2["Tail_detail_name"] = df2["Tail"].map(Pubchem_IUPAC_CID_Dict)

df2["head_type"]       = "PlantSpecies"
df2["tail_type"]       = "ChemicalEntity"
df2["relation"]        = "PlantSpecies_ChemicalEntity"
df2["head_detail_name"]= ""

df2['Tail_id_is'] = np.where(
    df2['Tail'].astype(str).str.isnumeric(),
    'Pubchem',
    None
)

df2["head_id_is"]      = ''

df2['relation_source'] = 'https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db'
df2.columns = df2.columns.str.lower()
df2 = reorder(df2)
print(f"PlantSpecies–Chemical rows: {len(df2):,}")
df2


PlantSpecies–Chemical rows: 880


,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Curcuma longa (turmeric),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...
1,Solanum torvum (Pea Eggplant),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,glycoalkaloid,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...
2,Rubia cordifolia(Indian Madder),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,124219,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",,Pubchem,https://github.com/asgarhussain/phytochemicals...
3,None,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...
4,Olea europaea (Olive),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,11652416,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",,Pubchem,https://github.com/asgarhussain/phytochemicals...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,extract,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...
876,Dracocephalum surmandinum,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,864,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,5-(dithiolan-3-yl)pentanoic acid,,Pubchem,https://github.com/asgarhussain/phytochemicals...
878,Ferula assa-foetida (Asafoetida),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,NaN,,None,https://github.com/asgarhussain/phytochemicals...


In [112]:
df2.to_csv(os.path.join(OUT_PATH, "ttd/Phytodata_Plantspecies_Chemical.csv"), index=False)
print("Saved: Phytodata_Plantspecies_Chemical.csv")
# del df2


Saved: Phytodata_Plantspecies_Chemical.csv


## §3 Phyto Data-1 — PlantSpecies–Disease


In [113]:
import sqlite3
import pandas as pd

DB_PATH = f'{BASE_PATH}other_sources/D1phyto_chemicals.db'   # change path if needed
# OUT_PATH = "phyto_kg_plantspecies_disease.csv"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM phyto_chemicals", conn)
conn.close()

# Build KG columns
df["Head"]      = df["Source_of_phytochemicals_(Name_of_Plant)"]
df["Relation"]  = "PlantSpecies_Disease"
df["Tail"]      = df["Type_of_cancer"]
df["Head_Type"] = "PlantSpecies"
df["Tail_Type"] = "Disease"
df["Source"]    = df["ADDITIONAL_SOURCES_OF_INFORMATION"]

COLS = [
    "Head", "Relation", "Tail",
    "Head_Type", "Tail_Type",
    "Source", "index",
    "Phytochemical_name/extract", "PMID"
]

out = df[COLS].dropna(subset=["Head", "Tail"], how="all").reset_index(drop=True)
# out.to_csv(OUT_PATH, index=False)
# print(f"Saved {len(out)} rows → {OUT_PATH}")
# print(out.head(3).to_string())
out

,Head,Relation,Tail,Head_Type,Tail_Type,Source,index,Phytochemical_name/extract,PMID
0,Curcuma longa (turmeric),PlantSpecies_Disease,"human hepatoma, human breast cancer",PlantSpecies,Disease,https://powo.science.kew.org/taxon/urn:lsid:ip...,0,Alpha-Turmerone,20438793.0
1,Solanum torvum (Pea Eggplant),PlantSpecies_Disease,breast cancer,PlantSpecies,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Sol...,1,Extract of Solanum torvum,34643844.0
2,Rubia cordifolia(Indian Madder),PlantSpecies_Disease,"human breast cancer, human ovarian cancer",PlantSpecies,Disease,None,2,Mollugin,23065756.0
3,None,PlantSpecies_Disease,breast cancer,PlantSpecies,Disease,None,3,Triphenylethylene-Coumarin Hybrid 5c,30947677.0
4,Olea europaea (Olive),PlantSpecies_Disease,breast cancer,PlantSpecies,Disease,None,4,(-)-Oleocanthal (OC),30781364.0
...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_Disease,"lung cancer, human colorectal adenocarcinoma, ...",PlantSpecies,Disease,https://pfaf.org/user/plant.aspx?latinname=Dio...,875,Diospyros lotus extract,19731038.0
876,Dracocephalum surmandinum,PlantSpecies_Disease,"human breast adenocarcinoma, erythromyeloblast...",PlantSpecies,Disease,https://powo.science.kew.org/taxon/urn:lsid:ip...,876,Dracocephalum surmandinum essential oil,20334155.0
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_Disease,breast cancer,PlantSpecies,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Ara...,877,Alpha-lipoic acid,20580704.0
878,Ferula assa-foetida (Asafoetida),PlantSpecies_Disease,breast cancer,PlantSpecies,Disease,https://pfaf.org/user/Plant.aspx?LatinName=Fer...,878,Essential Oil of Ferula assa-foetida,28900317.0


In [114]:
df3 = out
df3 = resolve_disease_tail(df3)

df3["head_type"]       = "PlantSpecies"
df3["tail_type"]       = "Disease"
df3["relation"]        = "PlantSpecies_Disease"
df3["head_detail_name"]= ""
df3["head_id_is"]      = ""

df3['Tail_id_is'] = np.where(
    df3['Tail'].isna(),
    None,
    np.where(
        df3['Tail'].str.startswith('DOID'),
        'DOID_ID',
        'MESH_ID'
    )
)

df3['relation_source'] = 'https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db'

df3.columns = df3.columns.str.lower()
df3 = reorder(df3)
print(f"PlantSpecies–Disease rows: {len(df3):,}")
df3

PlantSpecies–Disease rows: 1,448


,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Curcuma longa (turmeric),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
1,Solanum torvum (Pea Eggplant),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
2,Rubia cordifolia(Indian Madder),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
2,Rubia cordifolia(Indian Madder),PlantSpecies_Disease,PlantSpecies_Disease,DOID:2394,PlantSpecies,PlantSpecies,Disease,Disease,,ovarian cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
3,None,PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
876,Dracocephalum surmandinum,PlantSpecies_Disease,PlantSpecies_Disease,DOID:3458,PlantSpecies,PlantSpecies,Disease,Disease,,breast adenocarcinoma,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...
878,Ferula assa-foetida (Asafoetida),PlantSpecies_Disease,PlantSpecies_Disease,DOID:1612,PlantSpecies,PlantSpecies,Disease,Disease,,breast cancer,,DOID_ID,https://github.com/asgarhussain/phytochemicals...


In [115]:
df3.to_csv(os.path.join(OUT_PATH, "other_sources/D1_Phytodata_Plantspecies_Disease.csv"), index=False)
print("Saved: Phytodata_Plantspecies_Disease.csv")
# del df3


Saved: Phytodata_Plantspecies_Disease.csv


## §4 Phyto Data-2 — PlantSpecies–Chemical (GitHub)


In [121]:
IN_PATH  = f'{BASE_PATH}other_sources/D2Plant_compound.csv'  
# OUT_PATH = "plant_compound_kg.csv"
SOURCE   = "https://github.com/petermr/CEVOpen/blob/master/Data%20Analysis/Plant_compound.csv"

df = pd.read_csv(IN_PATH)

# Melt wide -> long (one row per plant-compound pair)
melted = df.melt(id_vars="PlantName", var_name="compound", value_name="count")

# Keep only pairs where compound is actually present
melted = melted[melted["count"] > 0].reset_index(drop=True)

# Build KG columns
out = pd.DataFrame({
    "Head":      melted["PlantName"],
    "Relation":  "PlantName_Chemical",
    "Tail":      melted["compound"],
    "Head_type": "Species",
    "Tail_type": "Chemical",
    "Source":    SOURCE,
})

# out.to_csv(OUT_PATH, index=True)
# print(f"Saved {len(out)} rows → {OUT_PATH}")
# print(out.head(5).to_string())
out

,Head,Relation,Tail,Head_type,Tail_type,Source
0,Origanum vulgare,PlantName_Chemical,thymol,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
1,Rosmarinus officinalis,PlantName_Chemical,thymol,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
2,Ocimum basilicum,PlantName_Chemical,thymol,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
3,Thymus vulgaris,PlantName_Chemical,thymol,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
4,Nigella sativa,PlantName_Chemical,thymol,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
...,...,...,...,...,...,...
313,Mentha pulegium,PlantName_Chemical,cinnamic acid,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
314,Cinnamon,PlantName_Chemical,cinnamic acid,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
315,Mentha longifolia,PlantName_Chemical,cinnamic acid,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...
316,Mentha piperita,PlantName_Chemical,cinnamic acid,Species,Chemical,https://github.com/petermr/CEVOpen/blob/master...


In [122]:
df4 = out
df4["Tail"] = df4["Tail"].map(Pubchem_Syn_fil_dict)
df4 = df4[~df4["Tail"].isna()]
df4["Tail"] = strip_float(df4["Tail"])
df4["Tail_detail_name"] = df4["Tail"].map(Pubchem_IUPAC_CID_Dict)

df4["head_type"]       = "PlantSpecies"
df4["tail_type"]       = "ChemicalEntity"
df4["relation"]        = "PlantSpecies_ChemicalEntity"
df4["head_detail_name"]= ""
df4["head_id_is"]      = ""

df4['Tail_id_is'] = np.where(
    df4['Tail'].astype(str).str.isnumeric(),
    'Pubchem',
    None
)

df4["relation_source"] = df4.get("Source", "Phytochemical_GitHub")

df4.columns = df4.columns.str.lower()
df4 = reorder(df4)
print(f"PlantSpecies–Chemical (Data-2) rows: {len(df4):,}")
df4


/tmp/ipykernel_1819267/1680774916.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4["Tail"] = strip_float(df4["Tail"])


PlantSpecies–Chemical (Data-2) rows: 81


/tmp/ipykernel_1819267/1680774916.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4["Tail_detail_name"] = df4["Tail"].map(Pubchem_IUPAC_CID_Dict)
/tmp/ipykernel_1819267/1680774916.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4["head_type"]       = "PlantSpecies"
/tmp/ipykernel_1819267/1680774916.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https:/

,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
39,Origanum vulgare,PlantName_Chemical,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
40,Rosmarinus officinalis,PlantName_Chemical,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
41,Ocimum basilicum,PlantName_Chemical,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
42,Thymus vulgaris,PlantName_Chemical,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
43,Nigella sativa,PlantName_Chemical,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,Nigella sativa,PlantName_Chemical,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
304,Mentha pulegium,PlantName_Chemical,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
305,Mentha longifolia,PlantName_Chemical,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...
306,Mentha piperita,PlantName_Chemical,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...


In [123]:

df4.to_csv(os.path.join(OUT_PATH, "other_sources/D2_Phytodata_PlantSpecies_Chemical_2.csv"), index=False)
print("Saved: D2_Phytodata_PlantSpecies_Chemical_2.csv")
# del df4


Saved: D2_Phytodata_PlantSpecies_Chemical_2.csv


In [125]:
#

## §5 Phyto Data-3 — PlantSpecies–Chemical (all_species_compound_species)


In [14]:
import pandas as pd

IN_PATH  = f'{BASE_PATH}other_sources/D3all_species_compound_data.csv'  

# IN_PATH  = "all_species_compound_data.csv"   # change path if needed
# OUT_PATH = "species_compound_kg.csv"

df = pd.read_csv(IN_PATH)

# Build KG columns
df.insert(0, "Head",      df["example_compound_name"])
df.insert(1, "Relation",  "Compound_Species")
df.insert(2, "Tail",      df["accepted_species"])
df.insert(3, "Head_Type", "Compound")
df.insert(4, "Tail_Type", "Species")
# Source column already exists in the original — keep it in place

# Drop rows where Head or Tail is missing
df = df.dropna(subset=["Head", "Tail"]).reset_index(drop=True)

# df.to_csv(OUT_PATH, index=False)
# print(f"Saved {len(df)} rows → {OUT_PATH}")
# print(df[["Head","Relation","Tail","Head_Type","Tail_Type","Source"]].head(3).to_string())
df

,Head,Relation,Tail,Head_Type,Tail_Type,Unnamed: 0,example_compound_name,Standard_SMILES,MAIP_SMILES,SMILES,...,NPclassif_superclass_results_0,NPclassif_pathway_results_0,NPclassif_class_results_1,NPclassif_pathway_results_1,NPclassif_superclass_results_1,NPclassif_class_results_2,NPclassif_class_results_3,NPclassif_pathway_results_2,NPclassif_superclass_results_2,Genus
0,acovenoside A,Compound_Species,Acokanthera laevigata,Compound,Species,0,acovenoside A,COC1C(O)C(C)OC(OC2CC(O)C3(C)C(CCC4C3CCC3(C)C(C...,COC1C(O)C(C)OC(OC2CC(O)C3(C)C(CCC4C3CCC3(C)C(C...,CC1C(C(C(C(O1)OC2CC3CCC4C(C3(C(C2)O)C)CCC5(C4(...,...,Steroids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acokanthera
1,(+)-linalool,Compound_Species,Acokanthera oblongifolia,Compound,Species,1,(+)-linalool,C=CC(C)(O)CCC=C(C)C,C=CC(C)(O)CCC=C(C)C,CC(=CCCC(C)(C=C)O)C,...,Monoterpenoids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acokanthera
2,(+)-linalyl acetate,Compound_Species,Acokanthera oblongifolia,Compound,Species,2,(+)-linalyl acetate,C=CC(C)(CCC=C(C)C)OC(C)=O,C=CC(C)(CCC=C(C)C)OC(C)=O,CC(=CCCC(C)(C=C)OC(=O)C)C,...,Monoterpenoids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acokanthera
3,(+)-α-pinene,Compound_Species,Acokanthera oblongifolia,Compound,Species,3,(+)-α-pinene,CC1=CCC2CC1C2(C)C,CC1=CCC2CC1C2(C)C,CC1=CCC2CC1C2(C)C,...,Monoterpenoids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acokanthera
4,(-)-beta-pinene,Compound_Species,Acokanthera oblongifolia,Compound,Species,4,(-)-beta-pinene,C=C1CCC2CC1C2(C)C,C=C1CCC2CC1C2(C)C,CC1(C2CCC(=C)C1C2)C,...,Monoterpenoids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acokanthera
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17313,Xysmalorin,Compound_Species,Xysmalobium undulatum,Compound,Species,18111,Xysmalorin,CC12CCC(OC3OC(CO)C(O)C(O)C3OC3OC(CO)C(O)C(O)C3...,CC12CCC(OC3OC(CO)C(O)C(O)C3OC3OC(CO)C(O)C(O)C3...,O=C1OCC(=C1)C2CCC3(O)C4CC=C5CC(OC6OC(CO)C(O)C(...,...,Steroids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Xysmalobium
17314,uzarigenin,Compound_Species,Xysmalobium undulatum,Compound,Species,18112,uzarigenin,CC12CCC(O)CC1CCC1C2CCC2(C)C(C3=CC(=O)OC3)CCC12O,CC12CCC(O)CC1CCC1C2CCC2(C)C(C3=CC(=O)OC3)CCC12O,CC12CCC(CC1CCC3C2CCC4(C3(CCC4C5=CC(=O)OC5)O)C)O,...,Steroids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Xysmalobium
17315,uzarin,Compound_Species,Xysmalobium undulatum,Compound,Species,18113,uzarin,CC12CCC(OC3OC(COC4OC(CO)C(O)C(O)C4O)C(O)C(O)C3...,CC12CCC(OC3OC(COC4OC(CO)C(O)C(O)C4O)C(O)C(O)C3...,O=C1OCC(=C1)C2CCC3(O)C4CCC5CC(OC6OC(COC7OC(CO)...,...,Steroids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Xysmalobium
17316,Centapicrin,Compound_Species,Zeltnera muehlenbergii,Compound,Species,18114,Centapicrin,C=CC1C(OC2OC(COc3cccc(C(=O)O)c3)C(O)C(OC(C)=O)...,C=CC1C(OC2OC(COc3cccc(C(=O)O)c3)C(O)C(OC(C)=O)...,O=C(O)C=1C=CC=C(OCC2OC(OC3OC=C4C(=O)OCCC4C3C=C...,...,Monoterpenoids,Terpenoids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zeltnera


In [17]:
#

In [18]:
df5 = df
# Original swaps Head↔Tail so species → Head, compound → Tail
df5[["Head", "Tail"]] = df5[["Tail", "Head"]]
df5
df5['Tail_temp'] = df5['Tail']
df5["Tail"] = df5["Tail_temp"].map(Pubchem_Syn_fil_dict)
df5 = df5[~df5["Tail"].isna()]
df5["Tail"] = strip_float(df5["Tail"])
df5["Tail_detail_name"] = df5["Tail"].map(Pubchem_IUPAC_CID_Dict)
df5
df5["head_type"]       = "PlantSpecies"
df5["tail_type"]       = "ChemicalEntity"
df5["relation"]        = "PlantSpecies_ChemicalEntity"
df5["head_detail_name"]= ""

df5['Tail_id_is'] = np.where(
    df5['Tail'].astype(str).str.isnumeric(),
    'Pubchem',
    None
)

df5["relation_source"] = 'https://github.com/alrichardbollans/PhytoChemicalDiversity/blob/master/collect_and_compile_data/collect_compound_data/outputs/all_species_compound_data.csv'

df5.columns = df5.columns.str.lower()
df5 = reorder(df5)
print(f"PlantSpecies–Chemical (Data-3) rows: {len(df5):,}")
df5

/tmp/ipykernel_1926692/2648018126.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5["Tail"] = strip_float(df5["Tail"])


PlantSpecies–Chemical (Data-3) rows: 5,630


/tmp/ipykernel_1926692/2648018126.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5["Tail_detail_name"] = df5["Tail"].map(Pubchem_IUPAC_CID_Dict)
/tmp/ipykernel_1926692/2648018126.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5["head_type"]       = "PlantSpecies"
/tmp/ipykernel_1926692/2648018126.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,tail_id_is,relation_source
2,Acokanthera oblongifolia,Compound_Species,PlantSpecies_ChemicalEntity,6999980,Compound,PlantSpecies,Species,ChemicalEntity,,"[(3S)-3,7-dimethylocta-1,6-dien-3-yl] acetate",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
6,Acokanthera oblongifolia,Compound_Species,PlantSpecies_ChemicalEntity,5281525,Compound,PlantSpecies,Species,ChemicalEntity,,"(3S,6E)-3,7,11-trimethyldodeca-1,6,10-trien-3-ol",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
7,Acokanthera oblongifolia,Compound_Species,PlantSpecies_ChemicalEntity,10261,Compound,PlantSpecies,Species,ChemicalEntity,,3-methyl-2-pent-2-enylcyclopent-2-en-1-one,Pubchem,https://github.com/alrichardbollans/PhytoChemi...
10,Acokanthera oblongifolia,Compound_Species,PlantSpecies_ChemicalEntity,133554300,Compound,PlantSpecies,Species,ChemicalEntity,,"3-[(1S,3S,5S,8S,10R,13S,14R,17S)-1,14-dihydrox...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
15,Acokanthera oblongifolia,Compound_Species,PlantSpecies_ChemicalEntity,244,Compound,PlantSpecies,Species,ChemicalEntity,,phenylmethanol,Pubchem,https://github.com/alrichardbollans/PhytoChemi...
...,...,...,...,...,...,...,...,...,...,...,...,...
17307,Xestaea lisianthoides,Compound_Species,PlantSpecies_ChemicalEntity,5281653,Compound,PlantSpecies,Species,ChemicalEntity,,"1,8-dihydroxy-2,6-dimethoxyxanthen-9-one",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
17310,Xysmalobium undulatum,Compound_Species,PlantSpecies_ChemicalEntity,10505,Compound,PlantSpecies,Species,ChemicalEntity,,"3-[(3S,10R,13R,14S)-3,14-dihydroxy-10,13-dimet...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
17312,Xysmalobium undulatum,Compound_Species,PlantSpecies_ChemicalEntity,321971,Compound,PlantSpecies,Species,ChemicalEntity,,"3-[14-hydroxy-10,13-dimethyl-3-[3,4,5-trihydro...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...
17313,Xysmalobium undulatum,Compound_Species,PlantSpecies_ChemicalEntity,208007,Compound,PlantSpecies,Species,ChemicalEntity,,"3-[(3S,8R,9S,10R,13R,14S,17R)-3-[(2R,3R,4S,5S,...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...


In [21]:
df5.to_csv(os.path.join(OUT_PATH, "other_sources/D3_Phytodata_PlantSpecies_Chemical_3.csv"), index=False)
print("Saved: Phytodata_PlantSpecies_Chemical_3.csv")
# del df5

Saved: Phytodata_PlantSpecies_Chemical_3.csv


## §6 Ayurvedic Data-1 — PlantSpecies–Disease


In [22]:
import json
import pandas as pd


IN_PATH  = f'{BASE_PATH}other_sources/D4Database.json' 

SOURCE   = "https://github.com/kuralamuthan300/ayurvedic-medicine-catalogue/blob/master/Database.json"

with open(IN_PATH) as f:
    data = json.load(f)

rows = []
for med_key, med in data["Medicine"].items():
    botanical = med["botanicalname"]
    parts     = med["parts"].strip()
    noofuses  = med["noofuses"]
    plantname = med["plantname"]
    dosage    = med["dosage"]
    uses      = med["therapeuticuses"].strip().strip('"').split()

    # Multi-word diseases: re-join using noofuses to split correctly
    # therapeuticuses is space-separated; use noofuses to know how many diseases
    # But some diseases are multi-word (e.g. "general debility", "renal calculi")
    # — split by noofuses count isn't reliable, so we parse the known multi-word ones
    # Instead: split on spaces but collapse known multi-word terms
    raw = med["therapeuticuses"].strip().strip('"').strip('"')

    # Known multi-word disease terms in this dataset
    MULTIWORD = [
        "general debility", "rheumatic ailments", "bleeding disorder",
        "burning micturition", "bone disorders", "eye diseases",
        "peptic ulcer", "skin diseases", "liver diseases", "renal calculi",
        "pitha humour diseases", "loss of appetite", "losso of appetite",
        "worm infestations", "male impotency", "pitha diseases",
        "mukkatra diseases", "kabam diseases", "fever with shivering",
        "fever with rigour", "alcohol addiction", "body heat",
        "dry mouth", "chronic dysentery", "chronic vatha diseases",
        "diabetes mellitus", "excessive thirst", "body pain",
        "lack of appetite", "reduced appetite",
        "loss of taste", "Loss of taste"
    ]
    # Replace multi-word terms with underscore-joined placeholders
    temp = raw
    for term in MULTIWORD:
        temp = temp.replace(term, term.replace(" ", "_"))

    diseases = [d.replace("_", " ") for d in temp.split()]

    for disease in diseases:
        rows.append({
            "Head":       botanical,
            "Relation":   "PlantSpecies_Disease",
            "Tail":       disease,
            "Head_Type":  "PlantSpecies",
            "Tail_Type":  "Disease",
            "Source":     SOURCE,
            "Medicine":   med_key,
            "Parts":      parts,
            "No of Uses": noofuses,
            "Plant Name": plantname,
            "Dosage":     dosage,
        })

out = pd.DataFrame(rows)
# out.to_csv(OUT_PATH, index=True)
# print(f"Saved {len(out)} rows → {OUT_PATH}")
# print(out.head(5).to_string())
out

,Head,Relation,Tail,Head_Type,Tail_Type,Source,Medicine,Parts,No of Uses,Plant Name,Dosage
0,Withania somnifera,PlantSpecies_Disease,pain,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,amukkara,Root,7,amukkara,Powder 3-6 g
1,Withania somnifera,PlantSpecies_Disease,fever,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,amukkara,Root,7,amukkara,Powder 3-6 g
2,Withania somnifera,PlantSpecies_Disease,eczema,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,amukkara,Root,7,amukkara,Powder 3-6 g
3,Withania somnifera,PlantSpecies_Disease,anemia,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,amukkara,Root,7,amukkara,Powder 3-6 g
4,Withania somnifera,PlantSpecies_Disease,general debility,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,amukkara,Root,7,amukkara,Powder 3-6 g
...,...,...,...,...,...,...,...,...,...,...,...
143,Trigonella foenum-graceum L,PlantSpecies_Disease,excessive thirst,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,venthayam,Seed,2,venthayam,Powder 3-6 g
144,Trigonella foenum-graceum L,PlantSpecies_Disease,leucorrhea,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,venthayam,Seed,2,venthayam,Powder 3-6 g
145,Aegle marmelo,PlantSpecies_Disease,fainting,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,vilvam,Root,3,vilvam,Decoction 30-50 ml twice daily
146,Aegle marmelo,PlantSpecies_Disease,fever,PlantSpecies,Disease,https://github.com/kuralamuthan300/ayurvedic-m...,vilvam,Root,3,vilvam,Decoction 30-50 ml twice daily


In [23]:
df6 = out

df6 = resolve_disease_tail(df6)

df6["head_detail_name"]= ""
df6["head_id_is"]      = ""
# df6["tail_id_is"]      = tail_id_is(df6["Tail"])
df6["relation_source"] = df6.get("Source", "Ayurvedic")

df6['Tail_id_is'] = np.where(
    df6['Tail'].isna(),
    None,
    np.where(
        df6['Tail'].str.startswith('DOID'),
        'DOID_ID',
        'MESH_ID'
    )
)

df6.columns = df6.columns.str.lower()
df6 = reorder(df6)
print(f"Ayurvedic PlantSpecies–Disease rows: {len(df6):,}")
df6


Ayurvedic PlantSpecies–Disease rows: 39


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
3,Withania somnifera,PlantSpecies_Disease,DOID:2355,PlantSpecies,Disease,,anemia,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
7,Ficus racemosa,PlantSpecies_Disease,DOID:12384,PlantSpecies,Disease,,dysentery,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
17,Glycyrrhiza glabra,PlantSpecies_Disease,C0030920,PlantSpecies,Disease,,peptic ulcer,,MESH_ID,https://github.com/kuralamuthan300/ayurvedic-m...
18,Glycyrrhiza glabra,PlantSpecies_Disease,DOID:12306,PlantSpecies,Disease,,vitiligo,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
22,Indigofera tinctoria,PlantSpecies_Disease,C0030920,PlantSpecies,Disease,,peptic ulcer,,MESH_ID,https://github.com/kuralamuthan300/ayurvedic-m...
25,Myristica fragrans,PlantSpecies_Disease,DOID:2841,PlantSpecies,Disease,,asthma,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
32,Tinospora cordifolia,PlantSpecies_Disease,DOID:0050127,PlantSpecies,Disease,,sinusitis,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
34,Tinospora cordifolia,PlantSpecies_Disease,DOID:10763,PlantSpecies,Disease,,hypertension,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
35,Cuminum cyminum,PlantSpecies_Disease,DOID:12384,PlantSpecies,Disease,,dysentery,,DOID_ID,https://github.com/kuralamuthan300/ayurvedic-m...
40,Cuminum cyminum,PlantSpecies_Disease,C0030920,PlantSpecies,Disease,,peptic ulcer,,MESH_ID,https://github.com/kuralamuthan300/ayurvedic-m...


In [24]:
df6.to_csv(os.path.join(OUT_PATH, "other_sources/D4_Phytodata_Ayurvedic_PlantSpecies_Disease.csv"), index=False)
print("Saved: Phytodata_Ayurvedic_PlantSpecies_Disease.csv")
# del df6


Saved: Phytodata_Ayurvedic_PlantSpecies_Disease.csv


## §7 Ayurvedic Data-2 — PlantSpecies–Chemical


In [12]:
#rghj

In [15]:
import pandas as pd
import re

IN_PATH  = f'{BASE_PATH}other_sources/D5english.xlsx'
# OUT_PATH = "english_kg.csv"
SOURCE   = "https://github.com/akshachrya/AyurvedicPlantClassifier/blob/main/data/english.xlsx"

# Skip sub-header row (row index 1)
df = pd.read_excel(IN_PATH, header=0, skiprows=[1])
df = df.dropna(subset=["Scientific name", "compounds"])

def parse_compounds(text):
    text = re.sub(
        r"(?i)(bioactive compounds[,\s]*including|"
        r"bioactive compounds[,\s]*|"
        r"azadirachtin and the others are|"
        r"including|and other photochemicals\.?|"
        r"and other[\w\s]*\.?)",
        ",", text
    )
    parts = re.split(r"[,;]", text)
    compounds = []
    for p in parts:
        p = p.strip().strip(".").strip()
        p = re.sub(r"(?i)^and\s+", "", p).strip()
        if p and len(p) > 1:
            compounds.append(p)
    return compounds

rows = []
for _, row in df.iterrows():
    for compound in parse_compounds(str(row["compounds"])):
        rows.append({
            "Head":                     row["Scientific name"].strip(),
            "Relation":                 "PlantSpecies_Chemical",
            "Tail":                     compound,
            "Head_Type":                "PlantSpecies",
            "Tail_Type":                "Chemical",
            "Source":                   SOURCE,
            "Plant part":               row["Plant part"],
            "common name":              row["common name"],
            "Family":                   row["Family"],
            "places found":             row["places found"],
            "coordinate":               row["coordinate"],
            "medicines":                row["medicines"],
            "Morphological appearance": row["Morphological appearance"],
            "Unnamed: 9":               row["Unnamed: 9"],
            "Unnamed: 10":              row["Unnamed: 10"],
        })

out = pd.DataFrame(rows).reset_index(drop=True)

out

,Head,Relation,Tail,Head_Type,Tail_Type,Source,Plant part,common name,Family,places found,coordinate,medicines,Morphological appearance,Unnamed: 9,Unnamed: 10
0,Azadirachta indica,PlantSpecies_Chemical,nimbolinin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
1,Azadirachta indica,PlantSpecies_Chemical,nimbin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
2,Azadirachta indica,PlantSpecies_Chemical,nimbidin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
3,Azadirachta indica,PlantSpecies_Chemical,nimbidol,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
4,Azadirachta indica,PlantSpecies_Chemical,sodium nimbinate,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
5,Azadirachta indica,PlantSpecies_Chemical,gedunin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
6,Azadirachta indica,PlantSpecies_Chemical,salannin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
7,Azadirachta indica,PlantSpecies_Chemical,quercetin,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",NaN,"immunomodulatory, anti-inflammatory, antiulcer...","dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis..."
8,Aloe barbadensis miller,PlantSpecies_Chemical,vitamins,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Aloe vera,Aloe Vera,Asphodelaceae,Native to Arabian Peninsula. Commonly grown in...,NaN,"1. Soothing and healing properties, often used...","Fleshy, green leaves",50-100 cm long,long and tapering leaves filled with clear gel...
9,Aloe barbadensis miller,PlantSpecies_Chemical,minerals,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Aloe vera,Aloe Vera,Asphodelaceae,Native to Arabian Peninsula. Commonly grown in...,NaN,"1. Soothing and healing properties, often used...","Fleshy, green leaves",50-100 cm long,long and tapering leaves filled with clear gel...

In [16]:
df7 = out

df7["Tail"] = df7["Tail"].str.strip()

# Map via lowercase synonym dict (compound names are mixed case)
df7["Tail_ID"] = df7["Tail"].str.lower().map(Pubchem_Syn_fil_dict_lower)
df7 = df7[~df7["Tail_ID"].isna()]
df7[["Tail_ID", "Tail"]] = df7[["Tail", "Tail_ID"]]
df7["Tail"] = strip_float(df7["Tail"])
df7["Tail_detail_name"] = df7["Tail"].map(Pubchem_IUPAC_CID_Dict)

df7["head_type"]       = "PlantSpecies"
df7["tail_type"]       = "ChemicalEntity"
df7["relation"]        = "PlantSpecies_ChemicalEntity"
df7["head_detail_name"]= ""
df7["head_id_is"]      = ""
df7

/tmp/ipykernel_1929979/1045633900.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df7[["Tail_ID", "Tail"]] = df7[["Tail", "Tail_ID"]]
/tmp/ipykernel_1929979/1045633900.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df7["Tail"] = strip_float(df7["Tail"])
/tmp/ipykernel_1929979/1045633900.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata

,Head,Relation,Tail,Head_Type,Tail_Type,Source,Plant part,common name,Family,places found,...,Morphological appearance,Unnamed: 9,Unnamed: 10,Tail_ID,Tail_detail_name,head_type,tail_type,relation,head_detail_name,head_id_is
1,Azadirachta indica,PlantSpecies_Chemical,108058,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",...,"dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis...",nimbin,"methyl (1S,2R,3R,4R,8R,9S,10R,13R,15R)-2-acety...",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
5,Azadirachta indica,PlantSpecies_Chemical,12004512,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",...,"dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis...",gedunin,"[(1S,2R,4S,7S,8S,11R,12R,17R,19R)-7-(furan-3-y...",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
6,Azadirachta indica,PlantSpecies_Chemical,6437066,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",...,"dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis...",salannin,"[(1R,2S,4R,6R,9R,10R,11R,12S,14R,15R,18R)-14-a...",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
7,Azadirachta indica,PlantSpecies_Chemical,5280343,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Neem leaf,Neem,mahogany family Meliaceae,"Native to Indian subcontinant, Sri lanka, Thai...",...,"dark green, Very young leaves are reddish to p...",3-8 cm long.,"pinnate, slightly curved like sickle, with dis...",quercetin,"2-(3,4-dihydroxyphenyl)-3,5,7-trihydroxychrome...",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
12,Ocimum sanctum,PlantSpecies_Chemical,3314,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Tulsi leaf,"Tulsi, Holy basil",Lamiaceae,"Native to Indian subcontinent, also found in S...",...,"Green, aromatic leaves",2-4 cm long,"Leaves are elliptical or ovate, having serrate...",eugenol,2-methoxy-4-prop-2-enylphenol,PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
13,Ocimum sanctum,PlantSpecies_Chemical,2537,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Tulsi leaf,"Tulsi, Holy basil",Lamiaceae,"Native to Indian subcontinent, also found in S...",...,"Green, aromatic leaves",2-4 cm long,"Leaves are elliptical or ovate, having serrate...",camphor,"1,7,7-trimethylbicyclo[2.2.1]heptan-2-one",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
14,Cymbopogon citratus,PlantSpecies_Chemical,638011,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Lemon Grass,Lemongrass,Poaceae,"Native to South Asia, also popular in countrie...",...,green leaves,can grow upto 1 meter,Slender grass,Citral,"(2E)-3,7-dimethylocta-2,6-dienal",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
15,Cymbopogon citratus,PlantSpecies_Chemical,31253,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Lemon Grass,Lemongrass,Poaceae,"Native to South Asia, also popular in countrie...",...,green leaves,can grow upto 1 meter,Slender grass,myrcene,"7-methyl-3-methylideneocta-1,6-diene",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
16,Cymbopogon citratus,PlantSpecies_Chemical,637566,PlantSpecies,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,Lemon Grass,Lemongrass,Poaceae,"Native to South Asia, also popular in countrie...",...,green leaves,can grow upto 1 meter,Slender grass,geraniol,"(2E)-3,7-dimethylocta-2,6-dien-1-ol",PlantSpecies,ChemicalEntity,PlantSpecies_ChemicalEntity,,
18,Citrus limon,PlantSpecies_Chemical,22311,PlantSpec

In [17]:
df['Tail_id_is'] = 'Pubchem_ID'
df7["relation_source"] = df7.get("Source", "Ayurvedic")

df7.columns = df7.columns.str.lower()
df7 = reorder(df7)
print(f"Ayurvedic PlantSpecies–Chemical rows: {len(df7):,}")
df7


Ayurvedic PlantSpecies–Chemical rows: 16


/tmp/ipykernel_1929979/4013493534.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df7["relation_source"] = df7.get("Source", "Ayurvedic")


,head,relation,relation,tail,head_type,head_type,tail_type,tail_type,head_detail_name,tail_detail_name,head_id_is,relation_source
1,Azadirachta indica,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,108058,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"methyl (1S,2R,3R,4R,8R,9S,10R,13R,15R)-2-acety...",,https://github.com/akshachrya/AyurvedicPlantCl...
5,Azadirachta indica,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,12004512,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"[(1S,2R,4S,7S,8S,11R,12R,17R,19R)-7-(furan-3-y...",,https://github.com/akshachrya/AyurvedicPlantCl...
6,Azadirachta indica,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,6437066,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"[(1R,2S,4R,6R,9R,10R,11R,12S,14R,15R,18R)-14-a...",,https://github.com/akshachrya/AyurvedicPlantCl...
7,Azadirachta indica,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,5280343,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"2-(3,4-dihydroxyphenyl)-3,5,7-trihydroxychrome...",,https://github.com/akshachrya/AyurvedicPlantCl...
12,Ocimum sanctum,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,3314,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,2-methoxy-4-prop-2-enylphenol,,https://github.com/akshachrya/AyurvedicPlantCl...
13,Ocimum sanctum,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,2537,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"1,7,7-trimethylbicyclo[2.2.1]heptan-2-one",,https://github.com/akshachrya/AyurvedicPlantCl...
14,Cymbopogon citratus,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,638011,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"(2E)-3,7-dimethylocta-2,6-dienal",,https://github.com/akshachrya/AyurvedicPlantCl...
15,Cymbopogon citratus,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,31253,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"7-methyl-3-methylideneocta-1,6-diene",,https://github.com/akshachrya/AyurvedicPlantCl...
16,Cymbopogon citratus,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,637566,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,"(2E)-3,7-dimethylocta-2,6-dien-1-ol",,https://github.com/akshachrya/AyurvedicPlantCl...
18,Citrus limon,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,22311,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,,1-methyl-4-prop-1-en-2-ylcyclohexene,,https://github.com/akshachrya/AyurvedicPlantCl...


In [18]:
df7.to_csv(os.path.join(OUT_PATH, "other_sources/D5_Phytodata_Ayurvedic_PlantSpecies_Chemical.csv"), index=False)
print("Saved: Phytodata_Ayurvedic_PlantSpecies_Chemical.csv")
# del df7


Saved: Phytodata_Ayurvedic_PlantSpecies_Chemical.csv


# §8 IMPPAT Data-1 — PlantSpecies–Chemical
**Source:** `all_phytochemical_plant_data.csv`

In [20]:
IMPPAT1 = pd.read_csv(
    os.path.join(IMPPAT_PATH, "all_phytochemical_plant_data.csv")
)
IMPPAT1["Tail"] = IMPPAT1["Phytochemical name"]
IMPPAT1["Head"] = IMPPAT1["Indian medicinal plant"]

IMPPAT1["Tail_ID"] = (
    IMPPAT1["Tail"].map(Pubchem_Syn_fil_dict)
    .fillna(IMPPAT1["Tail"].str.lower().map(Pubchem_Syn_fil_dict_lower))
    .fillna(IMPPAT1["Tail"].map(Pubchem_IUPAC_CID_Dict_2))
)
IMPPAT1 = IMPPAT1[~IMPPAT1["Tail_ID"].isna()]
IMPPAT1[["Tail_ID", "Tail"]] = IMPPAT1[["Tail", "Tail_ID"]]
IMPPAT1["Tail"] = strip_float(IMPPAT1["Tail"])
IMPPAT1["Tail_detail_name"] = IMPPAT1["Tail"].map(Pubchem_IUPAC_CID_Dict)

IMPPAT1["head_type"]       = "PlantSpecies"
IMPPAT1["tail_type"]       = "ChemicalEntity"
IMPPAT1["relation"]        = "PlantSpecies_ChemicalEntity"
IMPPAT1["head_detail_name"]= ""
IMPPAT1["head_id_is"]      = ""
IMPPAT1["tail_id_is"]      = "Pubchem"
IMPPAT1["relation_source"] = "IMPPAT"

IMPPAT1.columns = IMPPAT1.columns.str.lower()
IMPPAT1 = reorder(IMPPAT1)
print(f"IMPPAT PlantSpecies–Chemical rows: {len(IMPPAT1):,}")

IMPPAT PlantSpecies–Chemical rows: 180,185


In [21]:
IMPPAT1 = IMPPAT1.drop_duplicates(subset=['head', 'relation', 'tail'], keep='first')
IMPPAT1

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Piper cubeba,PlantSpecies_ChemicalEntity,10992619,PlantSpecies,ChemicalEntity,,"[(1R,2S,5R,6S)-5-benzoyloxy-1,2,6-trihydroxycy...",,Pubchem,IMPPAT
1,Clinopodium vulgare,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT
2,Cupressus funebris,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT
3,Petiveria alliacea,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT
6,Torilis leptophylla,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT
...,...,...,...,...,...,...,...,...,...,...
184937,Farsetia jacquemontii,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT
184938,Lepidium didymum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT
184939,Lepidium sativum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT
184941,Lepidium virginicum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT


In [22]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/immpat/'

In [23]:
IMPPAT1.to_csv(os.path.join(OUT_PATH,"immpat", "Phytodata_IMPPAT_PlantSpecies_Chemical.csv"), index=False)
print("Saved: Phytodata_IMPPAT_PlantSpecies_Chemical.csv")
# del IMPPAT1

Saved: Phytodata_IMPPAT_PlantSpecies_Chemical.csv


# §9 IMPPAT Data-2 — PlantSpecies–Disease
**Source:** `all_therapeutic_data.csv`



In [25]:
IMPPAT2 = pd.read_csv(
    os.path.join(IMPPAT_PATH, "all_therapeutic_data.csv")
)
IMPPAT2["Tail"] = IMPPAT2["Therapeutic use"]
IMPPAT2["Head"] = IMPPAT2["Indian medicinal plant"]

IMPPAT2["Tail_detail_name"] = (
    IMPPAT2["Tail"].map(DOID_disname_DOID_dict)
    .fillna(IMPPAT2["Tail"].str.lower().map(MONDOMESH_2_des_dict))
    .fillna(IMPPAT2["Tail"].map(MESH_name_dict))
)
IMPPAT2 = IMPPAT2[~IMPPAT2["Tail_detail_name"].isna()]

IMPPAT2[["Tail", "Tail_detail_name"]] = IMPPAT2[["Tail_detail_name", "Tail"]]

# Bug 11 fix: was PlantSpecies_ChemicalEntity / ChemicalEntity
IMPPAT2["head_type"]       = "PlantSpecies"
IMPPAT2["tail_type"]       = "Disease"
IMPPAT2["relation"]        = "PlantSpecies_Disease"
IMPPAT2["head_detail_name"]= ""
IMPPAT2["head_id_is"]      = ""
IMPPAT2["tail_id_is"]      = tail_id_is(IMPPAT2["Tail"])
IMPPAT2["relation_source"] = "IMPPAT"

IMPPAT2.columns = IMPPAT2.columns.str.lower()
IMPPAT2 = reorder(IMPPAT2)
print(f"IMPPAT PlantSpecies–Disease rows: {len(IMPPAT2):,}")
IMPPAT2

IMPPAT PlantSpecies–Disease rows: 52,324


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
1,Abelmoschus esculentus,PlantSpecies_Disease,D001046,PlantSpecies,Disease,,Aphrodisiacs,,MESH_ID,IMPPAT
2,Abelmoschus esculentus,PlantSpecies_Disease,C0009443,PlantSpecies,Disease,,Common cold,,MESH_ID,IMPPAT
3,Abelmoschus esculentus,PlantSpecies_Disease,D065106,PlantSpecies,Disease,,Demulcents,,MESH_ID,IMPPAT
4,Abelmoschus esculentus,PlantSpecies_Disease,D003967,PlantSpecies,Disease,,Diarrhea,,MESH_ID,IMPPAT
7,Abelmoschus esculentus,PlantSpecies_Disease,D004232,PlantSpecies,Disease,,Diuretics,,MESH_ID,IMPPAT
...,...,...,...,...,...,...,...,...,...,...
89723,Zosima absinthifolia,PlantSpecies_Disease,C0013395,PlantSpecies,Disease,,Dyspepsia,,MESH_ID,IMPPAT
89724,Zosima orientalis,PlantSpecies_Disease,D003371,PlantSpecies,Disease,,Cough,,MESH_ID,IMPPAT
89725,Zosima orientalis,PlantSpecies_Disease,D003371,PlantSpecies,Disease,,Cough,,MESH_ID,IMPPAT
89730,Zygophyllum propinquum,PlantSpecies_Disease,D058633,PlantSpecies,Disease,,Antipyretics,,MESH_ID,IMPPAT


In [26]:
IMPPAT2 = IMPPAT2.drop_duplicates(subset=['head', 'relation', 'tail'], keep='first')
IMPPAT2

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
1,Abelmoschus esculentus,PlantSpecies_Disease,D001046,PlantSpecies,Disease,,Aphrodisiacs,,MESH_ID,IMPPAT
2,Abelmoschus esculentus,PlantSpecies_Disease,C0009443,PlantSpecies,Disease,,Common cold,,MESH_ID,IMPPAT
3,Abelmoschus esculentus,PlantSpecies_Disease,D065106,PlantSpecies,Disease,,Demulcents,,MESH_ID,IMPPAT
4,Abelmoschus esculentus,PlantSpecies_Disease,D003967,PlantSpecies,Disease,,Diarrhea,,MESH_ID,IMPPAT
7,Abelmoschus esculentus,PlantSpecies_Disease,D004232,PlantSpecies,Disease,,Diuretics,,MESH_ID,IMPPAT
...,...,...,...,...,...,...,...,...,...,...
89721,Ziziphus xylopyrus,PlantSpecies_Disease,D001252,PlantSpecies,Disease,,Astringents,,MESH_ID,IMPPAT
89723,Zosima absinthifolia,PlantSpecies_Disease,C0013395,PlantSpecies,Disease,,Dyspepsia,,MESH_ID,IMPPAT
89724,Zosima orientalis,PlantSpecies_Disease,D003371,PlantSpecies,Disease,,Cough,,MESH_ID,IMPPAT
89730,Zygophyllum propinquum,PlantSpecies_Disease,D058633,PlantSpecies,Disease,,Antipyretics,,MESH_ID,IMPPAT


In [27]:
IMPPAT2.to_csv(os.path.join(OUT_PATH,"immpat", "Phytodata_IMPPAT_PlantSpecies_Disease.csv"), index=False)
print("Saved: Phytodata_IMPPAT_PlantSpecies_Disease.csv")
del IMPPAT2

Saved: Phytodata_IMPPAT_PlantSpecies_Disease.csv


# §10 PhytoHUB — PlantSpecies–Chemical
**Source:** `PhytoHub/Phytohub.csv`

In [52]:
phytohub = pd.read_csv(os.path.join(PHYTOHUB_PATH, "Phytohub.csv"))
phytohub['Tail'] = phytohub['Chemical Name']
phytohub['Head'] = phytohub['Food sources']

phytohub['Tail_ID'] = phytohub['Tail'].map(Pubchem_Syn_fil_dict).fillna(phytohub['Head'].map(Pubchem_Syn_fil_dict_lower))
phytohub = phytohub[~phytohub['Tail_ID'].isna()]
phytohub[['Tail_ID','Tail']] = phytohub[['Tail','Tail_ID']]
phytohub["Tail"] = phytohub["Tail"].astype(str).str.replace(r'\.0$', '', regex=True)
# df[['Head','head']] = df[['head','Head']]
phytohub['Tail_detail_name'] = phytohub['Tail'].map(Pubchem_IUPAC_CID_Dict)
phytohub
phytohub['Relation'] = 'PlantSpecies_ChemicalEntity'
phytohub['Head_type'] = 'PlantSpecies'
phytohub['Tail_type'] = 'ChemicalEntity'
phytohub['Head_id_is'] = 'Pubchem_ID'
phytohub['Tail_id_is'] = ''
phytohub['Relation_source'] = 'Phytohub'
phytohub['head_detail_name'] = ''
phytohub.columns = phytohub.columns.str.lower()
desired_order = ['head', 'relation', 'tail','head_type','tail_type','head_detail_name', 'tail_detail_name','head_id_is','tail_id_is','relation_source']
phytohub = phytohub[desired_order]
phytohub = phytohub[~phytohub['head'].isna()]
phytohub


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Coffee/Green robusta coffee/Roasted coffee/Rob...,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
1,Coffee,PlantSpecies_ChemicalEntity,157010316,PlantSpecies,ChemicalEntity,,"(1R,4R,5R,7R,9R,10S,13R,15S)-15-hydroxy-7-[(2S...",Pubchem_ID,,Phytohub
6,Coffee,PlantSpecies_ChemicalEntity,165360154,PlantSpecies,ChemicalEntity,,"dipotassium;[(2R,3R,4R,5R,6R)-2-[[(1R,4S,7S,9S...",Pubchem_ID,,Phytohub
8,Rice,PlantSpecies_ChemicalEntity,188456,PlantSpecies,ChemicalEntity,,"(4aS,4bS,5S,7S,10aR)-7-ethenyl-5,10a-dihydroxy...",Pubchem_ID,,Phytohub
9,Arabica coffee/Coffee/Robusta coffee,PlantSpecies_ChemicalEntity,114778,PlantSpecies,ChemicalEntity,,"(1S,4S,12S,13R,16R,17R)-17-(hydroxymethyl)-12-...",Pubchem_ID,,Phytohub
...,...,...,...,...,...,...,...,...,...,...
2642,Chickpea/Red clover,PlantSpecies_ChemicalEntity,44257215,PlantSpecies,ChemicalEntity,,"3-(4-methoxyphenyl)-7-[(2S,4S,5R)-3,4,5-trihyd...",Pubchem_ID,,Phytohub
2643,Chickpea/Red clover,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",Pubchem_ID,,Phytohub
2644,Red clover,PlantSpecies_ChemicalEntity,5281803,PlantSpecies,ChemicalEntity,,"5,7-dihydroxy-3-(3-hydroxy-4-methoxyphenyl)chr...",Pubchem_ID,,Phytohub
2645,Red clover,PlantSpecies_ChemicalEntity,5281805,PlantSpecies,ChemicalEntity,,"3-(1,3-benzodioxol-5-yl)-7-hydroxychromen-4-one",Pubchem_ID,,Phytohub


In [53]:
phytohub['head'] = phytohub['head'].astype(str).str.split('/')
# Step 2: Explode into separate rows
phytohub = phytohub.explode('head')
# Step 3 (optional): Remove extra spaces
phytohub['head'] = phytohub['head'].str.strip()
phytohub

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Green robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Roasted coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
1,Coffee,PlantSpecies_ChemicalEntity,157010316,PlantSpecies,ChemicalEntity,,"(1R,4R,5R,7R,9R,10S,13R,15S)-15-hydroxy-7-[(2S...",Pubchem_ID,,Phytohub
...,...,...,...,...,...,...,...,...,...,...
2643,Chickpea,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",Pubchem_ID,,Phytohub
2643,Red clover,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",Pubchem_ID,,Phytohub
2644,Red clover,PlantSpecies_ChemicalEntity,5281803,PlantSpecies,ChemicalEntity,,"5,7-dihydroxy-3-(3-hydroxy-4-methoxyphenyl)chr...",Pubchem_ID,,Phytohub
2645,Red clover,PlantSpecies_ChemicalEntity,5281805,PlantSpecies,ChemicalEntity,,"3-(1,3-benzodioxol-5-yl)-7-hydroxychromen-4-one",Pubchem_ID,,Phytohub


In [59]:
phytohub = phytohub.drop_duplicates(subset=['head', 'relation', 'tail'], keep='first')
phytohub
phytohub.to_csv(os.path.join(OUT_PATH,"phytohub", "Phytodata_PhytoHUB_PlantSpecies_Chemical.csv"), index=False)
print("Saved: Phytodata_PhytoHUB_PlantSpecies_Chemical.csv")
# del phytohub
phytohub

Saved: Phytodata_PhytoHUB_PlantSpecies_Chemical.csv


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Green robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Roasted coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
0,Robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",Pubchem_ID,,Phytohub
1,Coffee,PlantSpecies_ChemicalEntity,157010316,PlantSpecies,ChemicalEntity,,"(1R,4R,5R,7R,9R,10S,13R,15S)-15-hydroxy-7-[(2S...",Pubchem_ID,,Phytohub
...,...,...,...,...,...,...,...,...,...,...
2643,Chickpea,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",Pubchem_ID,,Phytohub
2643,Red clover,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",Pubchem_ID,,Phytohub
2644,Red clover,PlantSpecies_ChemicalEntity,5281803,PlantSpecies,ChemicalEntity,,"5,7-dihydroxy-3-(3-hydroxy-4-methoxyphenyl)chr...",Pubchem_ID,,Phytohub
2645,Red clover,PlantSpecies_ChemicalEntity,5281805,PlantSpecies,ChemicalEntity,,"3-(1,3-benzodioxol-5-yl)-7-hydroxychromen-4-one",Pubchem_ID,,Phytohub


# §11 TTD — Chemical–Disease
**Source:** `TTD/drug_disease_mapping.csv`

In [67]:
## Raw file to preprocess

import pandas as pd
import re

# Path to the input text file
file_path = f"{BASE_PATH}/ttd/P1-05-Drug_disease.txt"
# Read the file content
with open(file_path, "r") as file:
    lines = file.readlines()

# Initialize variables
data = []
current_entry = {}
indicati_list = []

# Process each line
for line in lines:
    line = line.strip()
    if not line or line.startswith("Abbreviations") or line.startswith("-"):
        continue

    match = re.match(r"^(TTDDRUID|DRUGNAME|INDICATI)\s+(.+)", line)
    if match:
        key, value = match.groups()
        value = value.strip()
        
        if key == "TTDDRUID":
            # Save previous entry if exists
            if current_entry:
                current_entry["INDICATI"] = indicati_list
                data.append(current_entry)
            # Start a new entry
            current_entry = {"TTDDRUID": value}
            indicati_list = []
        
        elif key == "DRUGNAME":
            current_entry["DRUGNAME"] = value

        elif key == "INDICATI":
            indicati_list.append(value)

# Append the last entry
if current_entry:
    current_entry["INDICATI"] = indicati_list
    data.append(current_entry)

# Convert data to a structured DataFrame
df = pd.DataFrame(data)

# Expand INDICATI into multiple columns
max_indications = df["INDICATI"].apply(len).max()
indication_columns = [f"INDICATI_{i+1}" for i in range(max_indications)]

df[indication_columns] = pd.DataFrame(df["INDICATI"].tolist(), index=df.index)
df = df.drop(columns=["INDICATI"])
df
df.to_csv(os.path.join(BASE_PATH,"ttd", "ttd_drug_disease_mapping.csv"), index=False)

In [72]:
TTD = pd.read_csv(f"{BASE_PATH}/ttd/ttd_drug_disease_mapping.csv")
TTD
# Skip the header-repeat first row; rename first two columns
TTD_cleaned = TTD.iloc[1:].reset_index(drop=True)
TTD_cleaned = TTD_cleaned.rename(columns={
    TTD_cleaned.columns[0]: "Head",
    TTD_cleaned.columns[1]: "Head_name",
})

# Combine all INDICATI_* columns into a single pipe-delimited Tail string
indication_cols = [c for c in TTD_cleaned.columns if c.startswith("INDICATI_")]
TTD_cleaned["Tail"] = (
    TTD_cleaned[indication_cols]
    .astype(str)
    .apply(lambda row: "|||||".join(v for v in row if v != "nan"), axis=1)
)

TTD_exp = (
    TTD_cleaned[["Head", "Head_name", "Tail"]]
    .assign(Tail=TTD_cleaned["Tail"].str.split("|||||", regex=False))
    .explode("Tail")
    .reset_index(drop=True)
)
TTD_exp = TTD_exp[TTD_exp["Tail"].str.strip() != ""]

# Clean Tail: strip tab-suffixed text and slash-suffixed text
TTD_exp["Tail"] = TTD_exp["Tail"].str.replace(r"\t.*", "", regex=True)
TTD_exp["Tail"] = TTD_exp["Tail"].str.replace(r"/.*",   "", regex=True)
TTD_exp["Tail"] = TTD_exp["Tail"].str.strip()

print(f"TTD exploded rows: {len(TTD_exp):,}")
TTD_exp

TTD exploded rows: 30,314


,Head,Head_name,Tail
0,DZB84T,Maralixibat,Pruritus
1,DZB84T,Maralixibat,Progressive familial intrahepatic cholestasis
2,DZB84T,Maralixibat,Alagille syndrome
3,DZA90G,BNT162b2,Coronavirus Disease 2019 (COVID-19)
4,DZ8DF0,Nedosiran,Primary hyperoxaluria type 1
...,...,...,...
30309,D00AJS,AIK11,Non-insulin dependent diabetes
30310,D00AIZ,SND-159,Mycobacterium infection
30311,D00AIS,AP-11014,Solid tumour
30312,D00AHV,SKL-PD,Central nervous system disease


In [73]:
# Resolve disease Tail
TTD_exp["Tail_detail_name"] = (
    TTD_exp["Tail"].map(DOID_disname_DOID_dict)
    .fillna(TTD_exp["Tail"].str.lower().map(MONDOMESH_2_des_dict))
    .fillna(TTD_exp["Tail"].map(MESH_name_dict))
)
TTD_exp = TTD_exp[~TTD_exp["Tail_detail_name"].isna()]
TTD_exp[["Tail", "Tail_detail_name"]] = TTD_exp[["Tail_detail_name", "Tail"]]

# Swap Head ↔ Head_name so canonical drug name goes to Head
TTD_exp[["Head", "Head_name"]] = TTD_exp[["Head_name", "Head"]]

# Resolve drug name → PubChem CID
TTD_exp["Head_ID"] = (
    TTD_exp["Head"].map(Pubchem_Syn_fil_dict)
    .fillna(TTD_exp["Head"].str.lower().map(Pubchem_Syn_fil_dict_lower))
)
TTD_exp = TTD_exp[~TTD_exp["Head_ID"].isna()]
TTD_exp[["Head_ID", "Head"]] = TTD_exp[["Head", "Head_ID"]]
TTD_exp["Head"] = strip_float(TTD_exp["Head"])
TTD_exp["Head_detail_name"] = TTD_exp["Head"].map(Pubchem_IUPAC_CID_Dict)

TTD_exp["head_type"]       = "ChemicalEntity"
TTD_exp["tail_type"]       = "Disease"
TTD_exp["relation"]        = "ChemicalEntity_Disease"
TTD_exp["head_id_is"]      = "Pubchem"
TTD_exp["tail_id_is"]      = tail_id_is(TTD_exp["Tail"])
TTD_exp["relation_source"] = "TTD"

TTD_exp.columns = TTD_exp.columns.str.lower()
TTD_exp = reorder(TTD_exp)
print(f"TTD Chemical–Disease rows: {len(TTD_exp):,}")
TTD_exp

/tmp/ipykernel_1819267/1562195992.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TTD_exp[["Tail", "Tail_detail_name"]] = TTD_exp[["Tail_detail_name", "Tail"]]
/tmp/ipykernel_1819267/1562195992.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TTD_exp[["Head", "Head_name"]] = TTD_exp[["Head_name", "Head"]]
/tmp/ipykernel_1819267/1562195992.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the

TTD Chemical–Disease rows: 5,999


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,9831643,ChemicalEntity_Disease,D011537,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Pruritus,Pubchem,MESH_ID,TTD
1,9831643,ChemicalEntity_Disease,C0268312,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Progressive familial intrahepatic cholestasis,Pubchem,MESH_ID,TTD
2,9831643,ChemicalEntity_Disease,DOID:9245,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Alagille syndrome,Pubchem,DOID_ID,TTD
11,129269915,ChemicalEntity_Disease,C0024305,ChemicalEntity,Disease,5-amino-3-[4-[[(5-fluoro-2-methoxybenzoyl)amin...,Non-hodgkin lymphoma,Pubchem,MESH_ID,TTD
48,77050711,ChemicalEntity_Disease,C0700345,ChemicalEntity,Disease,"(2R)-2-(2,4-difluorophenyl)-1,1-difluoro-3-(te...",Vulvovaginal Candidiasis,Pubchem,MESH_ID,TTD
...,...,...,...,...,...,...,...,...,...,...
30209,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,Disease,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Pubchem,MESH_ID,TTD
30238,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,Disease,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Pubchem,MESH_ID,TTD
30239,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,Disease,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Pubchem,MESH_ID,TTD
30279,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,Disease,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Pubchem,MESH_ID,TTD


In [74]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

In [76]:
TTD_exp = TTD_exp.drop_duplicates(subset=['head', 'relation', 'tail'], keep='first')
TTD_exp

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,9831643,ChemicalEntity_Disease,D011537,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Pruritus,Pubchem,MESH_ID,TTD
1,9831643,ChemicalEntity_Disease,C0268312,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Progressive familial intrahepatic cholestasis,Pubchem,MESH_ID,TTD
2,9831643,ChemicalEntity_Disease,DOID:9245,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Alagille syndrome,Pubchem,DOID_ID,TTD
11,129269915,ChemicalEntity_Disease,C0024305,ChemicalEntity,Disease,5-amino-3-[4-[[(5-fluoro-2-methoxybenzoyl)amin...,Non-hodgkin lymphoma,Pubchem,MESH_ID,TTD
48,77050711,ChemicalEntity_Disease,C0700345,ChemicalEntity,Disease,"(2R)-2-(2,4-difluorophenyl)-1,1-difluoro-3-(te...",Vulvovaginal Candidiasis,Pubchem,MESH_ID,TTD
...,...,...,...,...,...,...,...,...,...,...
30209,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,Disease,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Pubchem,MESH_ID,TTD
30238,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,Disease,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Pubchem,MESH_ID,TTD
30239,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,Disease,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Pubchem,MESH_ID,TTD
30279,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,Disease,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Pubchem,MESH_ID,TTD


In [77]:
TTD_exp.to_csv(
    f'{OUT_PATH}/ttd/Phytodata_TTD_Chemical_Disease.csv',
    index=False
)
print("Saved: Phytodata_TTD_Chemical_Disease.csv")
# del TTD, TTD_cleaned, TTD_exp

Saved: Phytodata_TTD_Chemical_Disease.csv
